In [3]:
import os 
import string
import nltk
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from langchain_core.documents import Document

In [6]:
nltk.download('punkt_tab')
# Thường thì bạn sẽ cần thêm 'punkt' và 'stopwords' nếu muốn làm sâu hơn
nltk.download('punkt')

[nltk_data] Downloading package punkt_tab to /home/vxh/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /home/vxh/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [4]:
data_dir = "./data"
documents = []

for filename in os.listdir(data_dir):
    if filename.endswith(".txt"):
        file_path = os.path.join(data_dir, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

            doc = Document(page_content=content, metadata={"source": filename})
            documents.append(doc)

print(f"Loaded {len(documents)} documents successfully")

Loaded 100 documents successfully


In [25]:
import re
import os
from rank_bm25 import BM25Okapi
import numpy as np
class BM25Retriever:
    def __init__(self, documents: list[Document]):
        """
        Khởi tạo BM25 với danh sách các Document.
        :param documents: List các đối tượng Document (có page_content và metadata)
        :param k: Số lượng kết quả top-K trả về
        """
        self.raw_documents = documents
        self.tokenized_corpus = [self._custom_tokenizer(doc.page_content) for doc in documents]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

    def _custom_tokenizer(self, text: str):
        text = text.lower()
        tokens = word_tokenize(text)
        tokens = [t for t in tokens if t not in string.punctuation]
        return tokens
    def invoke(self, query: str, k=None):
        """
        Truy vấn top-K tài liệu dựa trên điểm số BM25.
        """
        if k is not None:
            self.k = k
        tokenized_query = self._custom_tokenizer(query)
        doc_scores= self.bm25.get_scores(tokenized_query)
        top_n_indices = np.argsort(doc_scores)[::-1][:k]
        results = []
        for i in top_n_indices:
            score = doc_scores[i]
            if score > 0:
                doc = self.raw_documents[i]
                doc.metadata["bm25_score"] = float(score)
                results.append(doc)
        return results
    
    


bm25_retriever = BM25Retriever(documents=documents)
# results, scores = retriever.invoke("Quy định FPT-HR-001 về đi muộn")

In [26]:
test_queries = [
    "NXDOMAIN_1009",
    "ERR_CONNECTION_RESET_104",
    "MEM_LEAK_WARN_9901",
    "ThinkPad-T14-Gen3-21AH",
    "RESTful API"
]

for i, query in enumerate(test_queries):
    print(f"\n--- Câu hỏi {i+1}: '{query}' ---")
    results = bm25_retriever.invoke(query, k=2) # Chỉ in top 2 cho gọn
    
    if not results:
        print("-> Không tìm thấy kết quả nào!")
    
    for rank, res in enumerate(results):
        print(f"Top {rank+1} | Nguồn: {res.metadata['source']} | Điểm BM25: {res.metadata['bm25_score']}")
        # In 100 ký tự đầu tiên để kiểm tra
        print(f"Trích dẫn: {res.page_content[:100]}...\n")


--- Câu hỏi 1: 'NXDOMAIN_1009' ---
Top 1 | Nguồn: doc_063.txt | Điểm BM25: 2.61603748527231
Trích dẫn: Báo cáo sự cố hệ thống: Vào lúc 02:00 AM, monitor báo động với mã NXDOMAIN_1009. Nguyên nhân do việc...

Top 2 | Nguồn: doc_078.txt | Điểm BM25: 2.362904140964172
Trích dẫn: Quy trình khắc phục sự cố mạng: Khi người dùng phàn nàn về hiệu suất của Switch Cisco-C9200L-48P-4G,...


--- Câu hỏi 2: 'ERR_CONNECTION_RESET_104' ---
Top 1 | Nguồn: doc_026.txt | Điểm BM25: 2.5568565171012234
Trích dẫn: Thông báo cập nhật firmware cho Bàn phím cơ KCH-990-RGB. Phiên bản mới này sẽ cải thiện độ ổn định v...

Top 2 | Nguồn: doc_065.txt | Điểm BM25: 2.455646337759488
Trích dẫn: Hệ thống vừa ghi nhận sự cố liên quan đến ERR_CONNECTION_RESET_104. Đội ngũ kỹ thuật đang tiến hành ...


--- Câu hỏi 3: 'MEM_LEAK_WARN_9901' ---
Top 1 | Nguồn: doc_039.txt | Điểm BM25: 2.448850984876954
Trích dẫn: Khách hàng báo cáo thiết bị Ổ cứng SSD-Samsung-980-Pro-1TB không thể kết nối mạng. Qua kiểm tra ban ...

Top 2 